El objetivo de este Notebook es identificar y estudiar componentes conexas (clusters).

Cluster: conjunto de dominios interconectados mediante relaciones de infraestructura o similitud de contenido.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
cache  data  models  notebooks	README.md  requirements.txt  results


In [ ]:
import pickle
import networkx as nx
import pandas as pd


# Carga del grafo multimodal

In [ ]:
with open("results/graph_multimodal.gpickle", "rb") as f:    # Cargo el grafo final
    G = pickle.load(f)

G


In [ ]:
G.number_of_nodes(), G.number_of_edges()   # Compruebo el número de nodos y aristas


(2431, 3875)

# Análisis de clusters

Cluster= grupo de dominios conectados entre sí en el grafo

In [ ]:
components = list(nx.connected_components(G))  # Cúantos clusters independientes hay
len(components)


1767

In [ ]:
component_sizes = sorted([len(c) for c in components], reverse=True)  # Tamaño de los clusters más grandes
component_sizes[:10]


[49, 28, 21, 20, 18, 15, 14, 13, 13, 12]

In [ ]:
large_components = [c for c in components if len(c) >= 10]  # Número de clusters con más de 10 dominios
len(large_components)


17

# Caso A - Cluster infraestructural

In [ ]:
largest_component = max(components, key=len)
subG = G.subgraph(largest_component)
subG.number_of_nodes(), subG.number_of_edges()  # Mayor cluster, de cuantos nodos y aristas está formado


(49, 1176)

In [ ]:
list(subG.nodes())[:20]   # Dominios del cluster más grande


['ktu.iheart.com',
 'mix931.iheart.com',
 '953bull.iheart.com',
 'wild949.iheart.com',
 'klou.iheart.com',
 'rumba1065.iheart.com',
 'z100.iheart.com',
 'y94fm.iheart.com',
 '101kgb.iheart.com',
 'kiss108.iheart.com',
 '1061kissfm.iheart.com',
 '1029now.iheart.com',
 'star1043.iheart.com',
 'news.iheart.com',
 'star941fm.iheart.com',
 'altrock993.iheart.com',
 'kiisfm.iheart.com',
 'z100radio.iheart.com',
 'wjlbdetroit.iheart.com',
 'onairwithryan.iheart.com']

Se puede obersvar que el mayor cluser en el grafo está compuesto por 49 dominios pertenecientes a "iheart.com".

# Caso B - Similitud de contenido

Del Notebook 08 se puede observar que hay una arista (y por tanto un cluster) de contenido.

In [ ]:
(u, v, d), = [
    (u, v, d)
    for u, v, d in G.edges(data=True)
    if max(d.get("tfidf", 0), d.get("sbert", 0)) > 0    # Selección de las aristas que tengan contenido (TF-IDF, SBERT)
]

u, v, d


('Middle-east',
 'US_News',
 {'has_same_ip': False,
  'has_same_registrar': False,
  'tfidf': 0.9999237437592792,
  'sbert': 0.9999927,
  'weight': 0.9999927})

Compruebo que efectivamente solo hay una arista de contenido (formada por "Middle-east" y "US_News")

A continuación creo un subgrafo para poder aislar el caso de estudio y poder visualizarlo y analizarlo mejor.

In [ ]:
subG_content = G.subgraph([u, v])
subG_content.nodes(data=True), subG_content.edges(data=True)


(NodeDataView({'Middle-east': {}, 'US_News': {}}),
 EdgeDataView([('Middle-east', 'US_News', {'has_same_ip': False, 'has_same_registrar': False, 'tfidf': 0.9999237437592792, 'sbert': 0.9999927, 'weight': 0.9999927})]))

Se puede observar el alto valor de similitud obtenido tanto por TF-IDF como por SBERT. Esto significa que hay una presencia de contenido prácticamente duplicado o altamente redundante, lo que indica una posible reutilización de noticias entre dominios distintos.